# PPBench: data and training

Builds the PPBench dataset from the Pencil Puzzle Bench corpus, checks it, and trains TRM-Att on it with the recipe of
the released checkpoint (one A100 80GB, batch 768). To evaluate the released checkpoint, the dataset is enough: run
the notebook up to the checks, then copy `TRM/data/ppbench-6type` to the `DATA_PATH` of the evaluation notebooks.

In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install einops tqdm coolname pydantic argdantic wandb omegaconf hydra-core huggingface_hub numba
!pip -q install --no-deps ppbench==0.1.0
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)
!test -d TRM || git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels TRM
!git -C TRM checkout -q c011037
%cd TRM
%env WANDB_MODE=offline

## The dataset

The builder reads each puzzle and replays the moves of its solution with the puzzle engine of the `ppbench` package
(`pp_extract.js`). `pp_verify.js` checks label grids with the same engine.

In [ ]:
%%writefile dataset/build_ppbench_dataset.py
"""Builds the PPBench dataset of TRM from the Pencil Puzzle Bench corpus.

The golden puzzles are held out, then 100 puzzles per type (50 for tapa) are drawn for validation, and all three splits
keep 9x9 sudoku and 10x10 grids of the other types (sudoku is padded to 10x10). Each training puzzle gives --num-aug
examples: the puzzle itself and, for the others, the board after a random prefix of the solution's moves under a random
dihedral transform. A token is the full visible state of a cell (type, clue, answer mark, and the room walls for
heyawake); inputs keep the annotation dots of the replayed moves, labels are the solved board.

Layout: <out>/train/all__*.npy and <out>/test/{val,gold}_<type>__*.npy, one test set per split and type.
"""
import argparse, base64, collections, json, os, random, re, subprocess
import numpy as np

from common import PuzzleDatasetMetadata, dihedral_transform

TYPE_SIZE = {"sudoku": (9, 9), "lightup": (10, 10), "nurikabe": (10, 10), "shakashaka": (10, 10),
             "heyawake": (10, 10), "tapa": (10, 10)}
TYPES = list(TYPE_SIZE)  # also the order of the puzzle identifiers (0 is blank)
GRID = 10
PAD_TOKEN = 0
_XOR_KEY = b"ppbench"


def decrypt(enc):
    raw = base64.b64decode(enc)
    return json.loads(bytes(b ^ _XOR_KEY[i % len(_XOR_KEY)] for i, b in enumerate(raw)))


def clue_key(pid, y, x, rec):
    """Static, answer-independent part of a cell's token."""
    q = rec["qnum"][y][x]
    if pid == "sudoku":
        return ("g", q) if q > 0 else (".",)
    if pid in ("lightup", "shakashaka"):
        if q == -1: return ("open",)
        if q == -2: return ("wall",)
        return ("wall", q)
    if pid in ("nurikabe", "heyawake"):
        return ("n", q) if q >= 0 else (".",)
    if pid == "tapa":
        v = rec["qnums"][y][x]
        return ("c", tuple(v)) if v else (".",)
    raise ValueError(pid)


def mark_key(pid, y, x, s, keep_dots):
    """Answer part of a cell's token. keep_dots=False collapses annotations to empty."""
    qans, qsub, anum = s["qans"][y][x], s["qsub"][y][x], s["anum"][y][x]
    if pid == "sudoku":
        return ("d", anum) if anum > 0 else (".",)
    if qans:
        return ("t", qans) if pid == "shakashaka" else ("B",)
    if qsub and keep_dots:
        return ("o",)
    return (".",)


def base_grid(rec, pid, snapshot, keep_dots):
    """(H,W) object grid of cell states, in the puzzle's original frame, no room component."""
    H, W = rec["H"], rec["W"]
    g = np.empty((H, W), dtype=object)
    for y in range(H):
        for x in range(W):
            g[y, x] = (pid, clue_key(pid, y, x, rec), mark_key(pid, y, x, snapshot, keep_dots))
    return g


def rooms_from_walls(wallL, wallU, H, W):
    """Flood-fill the room partition implied by the internal walls."""
    rid = -np.ones((H, W), dtype=np.int64)
    nxt = 0
    for sy in range(H):
        for sx in range(W):
            if rid[sy, sx] >= 0:
                continue
            stack, rid[sy, sx] = [(sy, sx)], nxt
            while stack:
                y, x = stack.pop()
                if x + 1 < W and not wallL[y][x + 1] and rid[y, x + 1] < 0:
                    rid[y, x + 1] = nxt; stack.append((y, x + 1))
                if x - 1 >= 0 and not wallL[y][x] and rid[y, x - 1] < 0:
                    rid[y, x - 1] = nxt; stack.append((y, x - 1))
                if y + 1 < H and not wallU[y + 1][x] and rid[y + 1, x] < 0:
                    rid[y + 1, x] = nxt; stack.append((y + 1, x))
                if y - 1 >= 0 and not wallU[y][x] and rid[y - 1, x] < 0:
                    rid[y - 1, x] = nxt; stack.append((y - 1, x))
            nxt += 1
    return rid


def walls_from_rooms(rid):
    """Inverse of rooms_from_walls, applied after the dihedral transform."""
    H, W = rid.shape
    wl = np.zeros((H, W), dtype=np.int64); wu = np.zeros((H, W), dtype=np.int64)
    wl[:, 1:] = (rid[:, 1:] != rid[:, :-1])
    wu[1:, :] = (rid[1:, :] != rid[:-1, :])
    return wl, wu


def make_example(rec, pid, snapshot, tid, keep_dots, rooms0):
    """original-frame grid -> dihedral -> attach room walls in the new frame -> pad."""
    g = base_grid(rec, pid, snapshot, keep_dots)
    if tid:
        g = np.array(dihedral_transform(g, tid), dtype=object)
    if rooms0 is not None:
        rid = np.array(dihedral_transform(rooms0, tid), dtype=np.int64) if tid else rooms0
        wl, wu = walls_from_rooms(rid)
        code = wl + 2 * wu
        for y in range(g.shape[0]):
            for x in range(g.shape[1]):
                g[y, x] = g[y, x] + (int(code[y, x]),)
    if g.shape != (GRID, GRID):
        p = np.empty((GRID, GRID), dtype=object); p[:] = None
        p[:g.shape[0], :g.shape[1]] = g
        g = p
    return g


def build_grids(rec, rng, n_aug, augment):
    """[(input_grid, label_grid)] for one base puzzle."""
    pid, H, W = rec["pid"], rec["H"], rec["W"]
    rooms0 = None
    if pid == "heyawake":  # the room partition, without the walls on the outline or inside a room
        wl0 = np.array(rec["wallL"], dtype=bool)
        wu0 = np.array(rec["wallU"], dtype=bool)
        wl0[:, 0] = False; wu0[0, :] = False
        rooms0 = rooms_from_walls(wl0.astype(int).tolist(), wu0.astype(int).tolist(), H, W)
    snaps = [s["s"] for s in rec["snaps"]]
    out = []
    for j in range(n_aug if augment else 1):
        s = snaps[j] if j < len(snaps) else snaps[-1]
        tid = 0 if j == 0 else rng.randrange(8)
        out.append((make_example(rec, pid, s, tid, True, rooms0),
                    make_example(rec, pid, rec["solved"], tid, False, rooms0)))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", required=True, help="full_dataset.jsonl")
    ap.add_argument("--golden", required=True, help="golden_300.jsonl")
    ap.add_argument("--output-dir", default="data/ppbench-6type")
    ap.add_argument("--pzpr", required=True, help="path to pzpr.js")
    ap.add_argument("--extractor", default="dataset/pp_extract.js")
    ap.add_argument("--num-aug", type=int, default=11)
    ap.add_argument("--val-per-type", type=int, default=100)
    ap.add_argument("--val-tapa", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--work-dir", default="/tmp/ppwork")
    a = ap.parse_args()
    rng = random.Random(a.seed)
    os.makedirs(a.work_dir, exist_ok=True)

    golden = [json.loads(l) for l in open(a.golden) if l.strip()]

    def urlkey(u):
        if not u:
            return None
        return re.sub(r"^https?://", "", u).replace("pzv.jp/p.html?", "puzz.link/p?")

    golden_keys = set()
    for r in golden:
        for k in (urlkey(r.get("puzzlink_url")), urlkey(r.get("puzzle_url")), r.get("sort_key")):
            if k:
                golden_keys.add(k)

    pool = collections.defaultdict(list)
    n_src = n_hit = 0
    for line in open(a.source):
        if not line.strip():
            continue
        n_src += 1
        r = json.loads(line)
        if r["pid"] not in TYPE_SIZE:
            continue
        keys = {urlkey(r.get("puzzlink_url")), urlkey(r.get("puzzle_url")), r.get("sort_key")}
        if keys & golden_keys:
            n_hit += 1
            continue
        pool[r["pid"]].append(r)

    print(f"{n_src} source records, of which {n_hit} golden puzzles held out "
          f"({sum(1 for r in golden if r['pid'] in TYPE_SIZE)} golden puzzles of these types)")

    val, train = [], []
    for pid in TYPES:
        rs = sorted(pool[pid], key=lambda r: r["puzzlink_url"])
        rng.shuffle(rs)
        n = a.val_tapa if pid == "tapa" else a.val_per_type
        val += rs[:n]; train += rs[n:]

    size_ok = lambda r: (r["width"], r["height"]) == TYPE_SIZE[r["pid"]]
    splits = {"train": [r for r in train if size_ok(r)],
              "val":    [r for r in val if size_ok(r)],
              "gold":   [r for r in golden if r["pid"] in TYPE_SIZE and size_ok(r)]}
    print("puzzles after the grid-size filter:")
    for name, rs in splits.items():
        c = collections.Counter(r["pid"] for r in rs)
        print(f"  {name:5s} total={len(rs):6d}  " + "  ".join(f"{k}={c[k]}" for k in TYPES))

    for name, rs in splits.items():
        tin = os.path.join(a.work_dir, f"tasks_{name}.jsonl")
        tout = os.path.join(a.work_dir, f"ext_{name}.jsonl")
        n_nomove = 0
        with open(tin, "w") as f:
            for i, r in enumerate(rs):
                mv = decrypt(r["solution_enc"])["moves_full"]
                if not mv:
                    n_nomove += 1
                    continue
                cuts = [0] if name != "train" else \
                    [0] + sorted(rng.randrange(0, len(mv)) for _ in range(a.num_aug - 1))
                f.write(json.dumps({"i": i, "url": r["puzzlink_url"], "moves": mv,
                                    "cuts": cuts}) + "\n")
        if n_nomove:
            print(f"  [{name}] skipped {n_nomove} puzzles whose move list is empty")
        subprocess.run(["node", a.extractor, tin, tout], check=True,
                       env=dict(os.environ, PZPR_JS=os.path.abspath(a.pzpr)))

    vocab = {}

    def enc(g):
        out = np.empty(GRID * GRID, dtype=np.int32)
        for k, v in enumerate(g.ravel()):
            if v is None:
                out[k] = PAD_TOKEN
            else:
                t = vocab.get(v)
                if t is None:
                    t = vocab[v] = len(vocab) + 1
                out[k] = t
        return out

    grids = {}
    for name in ("train", "val", "gold"):
        rows, n_read, n_drop = [], 0, 0
        with open(os.path.join(a.work_dir, f"ext_{name}.jsonl")) as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                n_read += 1
                if "error" in r or not r.get("complete"):
                    n_drop += 1
                    continue
                for inp, lab in build_grids(r, rng, a.num_aug, name == "train"):
                    rows.append((r["pid"], enc(inp), enc(lab)))
        if n_drop:
            print(f"  [{name}] dropped {n_drop}/{n_read} puzzles whose stored moves do not solve them")
        grids[name] = rows

    V = len(vocab) + 1
    print(f"vocab_size = {V}, seq_len = {GRID * GRID}")

    tid_of = {t: i + 1 for i, t in enumerate(TYPES)}
    n_ids = len(TYPES) + 1

    os.makedirs(a.output_dir, exist_ok=True)
    for split_dir, members in (("train", [("all", "train")]),
                               ("test", [(f"{p}_{t}", p) for p in ("val", "gold") for t in TYPES])):
        out_dir = os.path.join(a.output_dir, split_dir)
        os.makedirs(out_dir, exist_ok=True)
        written, n_ex, n_groups = [], 0, 0
        for set_name, src in members:
            if split_dir == "train":
                items = grids["train"]
            else:
                t = set_name.split("_", 1)[1]
                items = [it for it in grids[src] if it[0] == t]
            if not items:
                continue
            per_group = a.num_aug if split_dir == "train" else 1
            assert len(items) % per_group == 0
            np.save(os.path.join(out_dir, f"{set_name}__inputs.npy"),
                    np.stack([i for _, i, _ in items]))
            np.save(os.path.join(out_dir, f"{set_name}__labels.npy"),
                    np.stack([l for _, _, l in items]))
            np.save(os.path.join(out_dir, f"{set_name}__puzzle_identifiers.npy"),
                    np.array([tid_of[p] for p, _, _ in items], dtype=np.int32))
            np.save(os.path.join(out_dir, f"{set_name}__puzzle_indices.npy"),
                    np.arange(len(items) + 1, dtype=np.int32))
            np.save(os.path.join(out_dir, f"{set_name}__group_indices.npy"),
                    np.arange(0, len(items) + 1, per_group, dtype=np.int32))
            written.append(set_name); n_ex += len(items); n_groups += len(items) // per_group
        meta = PuzzleDatasetMetadata(
            seq_len=GRID * GRID, vocab_size=V, pad_id=PAD_TOKEN, ignore_label_id=PAD_TOKEN,
            blank_identifier_id=0, num_puzzle_identifiers=n_ids, total_groups=n_groups,
            mean_puzzle_examples=1, total_puzzles=n_ex, sets=written)
        json.dump(meta.model_dump(), open(os.path.join(out_dir, "dataset.json"), "w"))
        print(f"  {split_dir}/  examples={n_ex}  groups={n_groups}  sets={len(written)}")
    json.dump(["<blank>"] + TYPES, open(os.path.join(a.output_dir, "identifiers.json"), "w"))
    json.dump({repr(k): v for k, v in vocab.items()},
              open(os.path.join(a.output_dir, "vocab.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile dataset/pp_extract.js
'use strict';
const fs = require('fs');
const pzpr = require(process.env.PZPR_JS);

function execinput(p, str) {
  const s = str.split(',');
  if (s[0] === 'mouse') {
    const b = s[1].split('+').pop();
    const m = b.match(/(left|right)(.*)/); if (!m) return;
    const rr = (m[2] || '').match(/x([0-9]+)/); const rep = rr ? +rr[1] : 1;
    const a = [];
    if (b.startsWith('left')) a.push('left'); else if (b.startsWith('right')) a.push('right');
    for (let i = 2; i < s.length; i++) a.push(+s[i]);
    for (let t = 0; t < rep; t++) p.mouse.inputPath.apply(p.mouse, a);
  } else if (s[0] === 'key') { s.shift(); p.key.inputKeys.apply(p.key, s); }
}
const sendMove = (p, mv) => { for (const a of mv.split(';')) execinput(p, a); };

const grid = (B, W, H, f) => {
  const g = [];
  for (let y = 0; y < H; y++) { const r = [];
    for (let x = 0; x < W; x++) r.push(f(B.getc(2 * x + 1, 2 * y + 1)));
    g.push(r); }
  return g;
};
const snap = (B, W, H) => ({
  qans: grid(B, W, H, c => c.qans | 0),
  qsub: grid(B, W, H, c => c.qsub | 0),
  anum: grid(B, W, H, c => (c.anum === undefined ? -1 : c.anum | 0)),
});

const [, , inPath, outPath] = process.argv;
const out = fs.createWriteStream(outPath);
let ok = 0, bad = 0, n = 0;

for (const line of fs.readFileSync(inPath, 'utf8').split('\n')) {
  if (!line.trim()) continue;
  const t = JSON.parse(line);
  try {
    const p = new pzpr.Puzzle().open(t.url);
    p.setMode('play');
    const B = p.board, W = B.cols, H = B.rows;

    const rec = { i: t.i, pid: p.pid, W, H,
      qnum:  grid(B, W, H, c => (c.qnum === undefined ? -1 : c.qnum | 0)),
      qnums: grid(B, W, H, c => (c.qnums ? c.qnums.slice().sort((a, b) => a - b) : [])),
      wallL: grid(B, W, H, c => { const b = B.getb(c.bx - 1, c.by); return (b && !b.isnull) ? (b.ques | 0) : 0; }),
      wallU: grid(B, W, H, c => { const b = B.getb(c.bx, c.by - 1); return (b && !b.isnull) ? (b.ques | 0) : 0; }),
      snaps: [] };

    const cuts = (t.cuts || []).slice().sort((a, b) => a - b);
    let ci = 0;
    while (ci < cuts.length && cuts[ci] <= 0) { rec.snaps.push({ k: cuts[ci], s: snap(B, W, H) }); ci++; }
    for (let j = 0; j < t.moves.length; j++) {
      sendMove(p, t.moves[j]);
      while (ci < cuts.length && cuts[ci] === j + 1) { rec.snaps.push({ k: cuts[ci], s: snap(B, W, H) }); ci++; }
    }
    rec.solved = snap(B, W, H);
    rec.complete = !!p.check(false).complete;
    out.write(JSON.stringify(rec) + '\n');
    ok++;
  } catch (e) {
    out.write(JSON.stringify({ i: t.i, error: String((e && e.message) || e) }) + '\n');
    bad++;
  }
  if ((++n) % 2000 === 0) process.stderr.write(`  ...${n}\r`);
}
out.end();
console.error(`extracted ok=${ok} failed=${bad}`);

In [ ]:
%%writefile dataset/pp_verify.js
'use strict';
const fs = require('fs');
const pzpr = require(process.env.PZPR_JS);
let ok = 0, bad = 0, err = 0;
for (const line of fs.readFileSync(process.argv[2], 'utf8').split('\n')) {
  if (!line.trim()) continue;
  const t = JSON.parse(line);
  try {
    const p = new pzpr.Puzzle().open(t.url); p.setMode('play'); const B = p.board;
    for (const s of t.sets) {
      const c = B.getc(2 * s[1] + 1, 2 * s[0] + 1);
      if (s[2] === 'anum') c.setAnum(s[3]); else c.setQans(s[3]);
    }
    if (p.check(false).complete) ok++; else { bad++; console.log('NOT COMPLETE', t.pid, t.url); }
  } catch (e) { err++; console.log('ERR', t.pid, String((e && e.message) || e)); }
}
console.log(`label == solved:  ok=${ok}  not_complete=${bad}  error=${err}`);

In [ ]:
import importlib.util, json, os, subprocess, sys

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

PPBENCH = os.path.dirname(importlib.util.find_spec("ppbench").origin)
PZPR = os.path.join(PPBENCH, "vendor", "pzprjs", "dist", "js", "pzpr.js")
GOLDEN = os.path.join(PPBENCH, "bundled", "golden_300.jsonl")
CORPUS = hf_hub_download("bluecoconut/pencil-puzzle-bench", "full_dataset.jsonl", repo_type="dataset",
                         revision="3ac6add0754f1458bf51aab160ca039667a67b38")

In [ ]:
!python dataset/build_ppbench_dataset.py --source {CORPUS} --golden {GOLDEN} --pzpr {PZPR} \
    --output-dir data/ppbench-6type --num-aug 11 --val-per-type 100 --val-tapa 50 --seed 0

## Checks

The engine accepts the label of every golden puzzle as a solution, the rooms of heyawake follow the dihedral
transforms, and sudoku is padded with 19 cells that the loss and the metrics ignore. The table counts the puzzles per
type in each split, with PTRM's training count (its training examples divided by 11).

In [ ]:
sys.path.insert(0, "dataset")
from build_ppbench_dataset import TYPES, make_example, rooms_from_walls
from common import dihedral_transform

golden = [r for r in map(json.loads, open("/tmp/ppwork/ext_gold.jsonl")) if "error" not in r and r.get("complete")]
urls = {t["i"]: t["url"] for t in map(json.loads, open("/tmp/ppwork/tasks_gold.jsonl"))}
tasks = []
for r in golden:
    rooms = rooms_from_walls(r["wallL"], r["wallU"], r["H"], r["W"]) if r["pid"] == "heyawake" else None
    label = make_example(r, r["pid"], r["solved"], 0, False, rooms)
    cells = []
    for y in range(r["H"]):
        for x in range(r["W"]):
            mark = label[y, x][2]
            if mark[0] == "B":
                cells.append([y, x, "qans", 1])
            elif mark[0] == "t":
                cells.append([y, x, "qans", int(mark[1])])
            elif mark[0] == "d":
                cells.append([y, x, "anum", int(mark[1])])
    tasks.append({"url": urls[r["i"]], "pid": r["pid"], "sets": cells})
with open("/tmp/ppwork/verify.jsonl", "w") as f:
    f.write("\n".join(map(json.dumps, tasks)))
out = subprocess.run(["node", "dataset/pp_verify.js", "/tmp/ppwork/verify.jsonl"], env=dict(os.environ, PZPR_JS=PZPR),
                     check=True, capture_output=True, text=True).stdout
assert "ok=%d  not_complete=0  error=0" % len(tasks) in out, out


def rooms_of(grid, H, W):
    """The room partition encoded by the wall bits of a heyawake grid."""
    left = [[grid[y, x][3] & 1 for x in range(W)] for y in range(H)]
    up = [[(grid[y, x][3] >> 1) & 1 for x in range(W)] for y in range(H)]
    return rooms_from_walls(left, up, H, W)


def same_partition(a, b):
    pairs = set(zip(a.ravel(), b.ravel()))
    return len(pairs) == len({u for u, _ in pairs}) == len({v for _, v in pairs})


for r in (r for r in golden if r["pid"] == "heyawake"):
    rooms = rooms_from_walls(r["wallL"], r["wallU"], r["H"], r["W"])
    for tid in range(8):
        grid = make_example(r, "heyawake", r["solved"], tid, False, rooms)
        assert same_partition(rooms_of(grid, r["H"], r["W"]), np.array(dihedral_transform(rooms, tid)))

for split in ("val", "gold"):
    for name in ("inputs", "labels"):
        assert ((np.load("data/ppbench-6type/test/%s_sudoku__%s.npy" % (split, name)) == 0).sum(1) == 19).all()

meta = json.load(open("data/ppbench-6type/train/dataset.json"))
ids = np.load("data/ppbench-6type/train/all__puzzle_identifiers.npy")
PTRM_TRAIN = {"sudoku": 7810, "lightup": 9504, "nurikabe": 15180, "shakashaka": 20702, "heyawake": 42108, "tapa": 3663}
pd.DataFrame({t: {"train": (ids == i + 1).sum() // 11, "PTRM train": PTRM_TRAIN[t] // 11,
                  "validation": len(np.load("data/ppbench-6type/test/val_%s__inputs.npy" % t)),
                  "golden": len(np.load("data/ppbench-6type/test/gold_%s__inputs.npy" % t))}
              for i, t in enumerate(TYPES)}).T

## Training

TRM's training script is patched in two places: the optimizer is AdamW, and the evaluation log adds the aggregate of
each split over the five reported types (`val_AGG`, `gold_AGG`). The model is evaluated every 1,000 epochs on the
twelve test sets, and its EMA weights are saved at each evaluation in `checkpoints/<project>/<run>/`; the released
checkpoint is the last one. The number of epochs gives about 255,000 optimizer steps (21,000 epochs at batch 768).
On a GPU with less than 80 GB, halve `BATCH`: the number of epochs follows, but the run departs from the recipe.

In [ ]:
%%writefile ppbench.patch
diff --git a/pretrain.py b/pretrain.py
index b9072e2..8720030 100644
--- a/pretrain.py
+++ b/pretrain.py
@@ -17,7 +17,7 @@ import coolname
 import hydra
 import pydantic
 from omegaconf import DictConfig
-from adam_atan2 import AdamATan2
+from torch.optim import AdamW as AdamATan2
 
 from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
 from utils.functions import load_model_class, get_model_source_path
@@ -82,6 +82,7 @@ class PretrainConfig(pydantic.BaseModel):
     ema: bool = False # use Exponential-Moving-Average
     ema_rate: float = 0.999 # EMA-rate
     freeze_weights: bool = False # If True, freeze weights and only learn the embeddings
+    report_types: List[str] = []  # types aggregated per split in the evaluation log (e.g. val_AGG)
 
 @dataclass
 class TrainState:
@@ -440,19 +441,21 @@ def evaluate(
                 dist.reduce(metric_values, dst=0)
 
             if rank == 0:
-                reduced_metrics = metric_values.cpu().numpy()
-                reduced_metrics = {
-                    set_name: {
-                        metric_name: reduced_metrics[set_id, metric_id]
-                        for metric_id, metric_name in enumerate(metric_keys)
-                    }
-                    for set_id, set_name in enumerate(set_ids)
-                }
-
-                # Postprocess
-                for set_name, m in reduced_metrics.items():
-                    count = m.pop("count")
-                    reduced_metrics[set_name] = {k: v / count for k, v in m.items()}
+                reduced_metrics, counts = {}, {}
+                for set_id, set_name in enumerate(set_ids):
+                    m = dict(zip(metric_keys, metric_values[set_id].cpu().numpy()))
+                    counts[set_name] = m.pop("count")
+                    reduced_metrics[set_name] = {k: v / counts[set_name] for k, v in m.items()}
+
+                # Aggregate of each split over the reported types, weighted by their numbers of puzzles
+                for split in sorted({s.split("_", 1)[0] for s in set_ids}):
+                    sets = [s for s in set_ids if s.startswith(split + "_") and s.split("_", 1)[1] in config.report_types]
+                    if sets:
+                        total = sum(counts[s] for s in sets)
+                        reduced_metrics[split + "_AGG"] = {k: sum(reduced_metrics[s][k] * counts[s] for s in sets) / total
+                                                           for k in reduced_metrics[sets[0]]}
+                for name in sorted(reduced_metrics):
+                    print("%-18s exact accuracy %6.2f%%" % (name, 100 * reduced_metrics[name]["exact_accuracy"]))
 
         # Run evaluators
         if rank == 0:

In [ ]:
!git checkout -q pretrain.py && git apply ppbench.patch

BATCH = 768
EPOCHS = round(255000 * BATCH / meta["total_groups"] / 1000) * 1000

In [ ]:
!python pretrain.py arch=trm data_paths="[data/ppbench-6type]" evaluators="[]" \
    +report_types="[sudoku,lightup,nurikabe,heyawake,tapa]" global_batch_size={BATCH} epochs={EPOCHS} \
    eval_interval=1000 lr=1e-4 puzzle_emb_lr=1e-4 weight_decay=1.0 puzzle_emb_weight_decay=1.0 arch.mlp_t=False \
    arch.pos_encodings=rope arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=6 ema=True